# Process and classify any supported dataset.

### Imports

In [1]:
import numpy as np
from sklearn.model_selection import cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.model_selection import StratifiedKFold, GroupKFold

from fnirsatlas.load_parser import load_parser
from fnirsatlas.processed_dataset import to_dataset
from fnirsatlas.feature_extraction import WindowedFeatureExtractor


### Process dataset

In [2]:
dataset_name = "bnci"

parser = load_parser(dataset_name)
epochs_dict = parser.get_epochs()
X, y, groups, info = to_dataset(epochs_dict)

Processing bnci dataset: 100%|██████████| 8/8 [00:07<00:00,  1.11it/s]


### Cross-subject classification

In [3]:
clf = Pipeline(
    [
        ("extractor", WindowedFeatureExtractor(n_windows=6, features=["mean"])),
        ("scaler", StandardScaler()),
        ("clf", LinearDiscriminantAnalysis(shrinkage="auto", solver="lsqr")),
    ]
)

subject_ids = list(dict.fromkeys(groups))
gkf = GroupKFold(n_splits=len(subject_ids))  # Leave-one-subject-out
cross_scores = cross_val_score(clf, X, y, groups=groups, cv=gkf, scoring="accuracy")
print(f"Cross-subject accuracy:  {np.mean(cross_scores):.2f}")

Cross-subject accuracy:  0.79


### Within-subject classification

In [4]:
sub_scores = []
for sub in subject_ids:
    sub_mask = groups == sub
    Xs, ys = X[sub_mask], y[sub_mask]
    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=0)
    sub_score = cross_val_score(clf, Xs, ys, cv=skf, scoring="accuracy").mean()
    sub_scores.append(sub_score)
    print(f"    Subject {sub}: {sub_score:.2f}")

print(f"\nWithin-subject accuracy: {np.mean(sub_scores):.2f}")

    Subject S01: 1.00
    Subject S02: 0.97
    Subject S03: 1.00
    Subject S04: 1.00
    Subject S05: 1.00
    Subject S06: 0.98
    Subject S07: 1.00
    Subject S08: 1.00

Within-subject accuracy: 0.99
